<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_CRCV_Securitization_ACTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# --- Helper function for display ---
def display_df(df, title=""):
    """Prints a DataFrame with a title for script-based output."""
    print(f"--- {title} ---")
    # Use to_string() to ensure the full DataFrame is printed
    print(df.to_string())
    print("\n" + "="*80 + "\n")

# ==============================================================================
# Cell 1: Steps 3 & 4 - Establish Net Positions
# ==============================================================================
print("### Step 3 & 4: Establish Net Positions ###\n")
print("As per Article 325g(1), CVR values are calculated on a net basis for each risk factor (issuer).\n")
print("The following DataFrame represents the final net positions for our portfolio.\n")


# Define the initial portfolio of net curvature risk positions
net_positions_data = {
    'Bucket': [5, 5, 5, 5, 5, 6, 6, 6, 6],
    'Issuer': ['Issuer A', 'Issuer B', 'Issuer C', 'Issuer D', 'Issuer E', 'Issuer F', 'Issuer G', 'Issuer H', 'Issuer I'],
    'CVR+': [-883, -4591, -4358, -884, -778, -384, -347, -236, -2243],
    'CVR-': [1621, 4061, 4421, 1743, 1594, 1091, 1215, 932, 2854]
}
net_positions_df = pd.DataFrame(net_positions_data)

display_df(net_positions_df, "Final Net Positions")

# ==============================================================================
# Cell 2: Steps 5 & 6 - Determine Base Curvature Correlations
# ==============================================================================
print("### Steps 5 & 6: Determine Base Curvature Correlations (Medium Scenario) ###\n")
print("Curvature correlations are the square of the corresponding delta correlations (Article 325ay(5)).\n")

# --- Intra-Bucket Correlation (rho) ---
# Delta correlation for different names in a CTP bucket is 35%
delta_rho_ctp = 0.35
curvature_rho_medium = delta_rho_ctp**2
print(f"Base Intra-Bucket Delta Correlation (rho_delta): {delta_rho_ctp:.2%}")
print(f"Base Intra-Bucket Curvature Correlation (rho_curvature): {curvature_rho_medium:.4f} or {curvature_rho_medium:.2%}\n")


# --- Cross-Bucket Correlation (gamma) ---
# Delta correlation between Bucket 5 (Consumer) and 6 (Technology) is 25%
delta_gamma_ctp = 0.25
curvature_gamma_medium = delta_gamma_ctp**2
print(f"Base Cross-Bucket Delta Correlation (gamma_delta): {delta_gamma_ctp:.2%}")
print(f"Base Cross-Bucket Curvature Correlation (gamma_curvature): {curvature_gamma_medium:.4f} or {curvature_gamma_medium:.2%}")
print("\n" + "="*80 + "\n")

# ==============================================================================
# Cell 3: Steps 7 & 8 - Bucket Capital Calculation Logic
# ==============================================================================
print("### Steps 7 & 8: Bucket-Level Capital Calculation ###\n")

def psi(val1, val2):
    """Safeguard function from Article 325g(4). Returns 0 if both inputs are negative, otherwise 1."""
    return np.where((val1 < 0) & (val2 < 0), 0, 1)

def calculate_bucket_capital_components(bucket_df, correlation):
    """
    Calculates K_b+, K_b-, the final K_b, and the selected scenario for a single bucket.
    This version uses a robust, vectorized approach for the correlation term.
    """
    # --- Helper to calculate one scenario (K+ or K-) ---
    def get_k_scenario(cvr_vector, rho_val):
        sum_sq = np.sum(np.maximum(cvr_vector, 0)**2)

        # Vectorized correlation term calculation
        outer_prod = np.outer(cvr_vector, cvr_vector)
        psi_matrix = ~((cvr_vector[:, None] < 0) & (cvr_vector[None, :] < 0))
        full_matrix = rho_val * outer_prod * psi_matrix
        # Sum only the off-diagonal elements
        corr_term = np.sum(full_matrix) - np.sum(np.diag(full_matrix))

        return np.sqrt(max(0, sum_sq + corr_term))

    # --- Upward & Downward Scenarios ---
    kb_plus = get_k_scenario(bucket_df['CVR+'].values, correlation)
    kb_minus = get_k_scenario(bucket_df['CVR-'].values, correlation)

    # --- Final K_b and Scenario Selection ---
    kb_final = max(kb_plus, kb_minus)

    if kb_plus == kb_minus:
        selected_scenario = 'Upward' if bucket_df['CVR+'].sum() > bucket_df['CVR-'].sum() else 'Downward'
    else:
        selected_scenario = 'Upward' if kb_plus > kb_minus else 'Downward'

    return kb_plus, kb_minus, kb_final, selected_scenario

# Calculate for each bucket using medium correlation
bucket_results = {}
for bucket_id, group in net_positions_df.groupby('Bucket'):
    print(f"Calculating for Bucket {bucket_id}...")
    k_plus, k_minus, k_final, scenario = calculate_bucket_capital_components(group, curvature_rho_medium)
    bucket_results[bucket_id] = {
        'K_b+': k_plus,
        'K_b-': k_minus,
        'K_b': k_final,
        'Selected Scenario': scenario
    }
    print(f"  K_b+ = {k_plus:,.2f}")
    print(f"  K_b- = {k_minus:,.2f}")
    print(f"  Final K_b = {k_final:,.2f}")
    print(f"  Selected Scenario: {scenario}\n")

display_df(pd.DataFrame(bucket_results).T.reset_index().rename(columns={'index': 'Bucket'}), "Summary of Bucket Capital (Medium Scenario)")

# ==============================================================================
# Cell 4: Step 9 - Determine Bucket Sums (S_b)
# ==============================================================================
print("### Step 9: Determine Bucket Sums (S_b) ###\n")
print("Calculating S_b based on the selected scenario for each bucket (Article 325g(6)).\n")

bucket_sums = {}
for bucket_id, result in bucket_results.items():
    bucket_df = net_positions_df[net_positions_df['Bucket'] == bucket_id]
    if result['Selected Scenario'] == 'Upward':
        s_b = bucket_df['CVR+'].sum()
    else:
        s_b = bucket_df['CVR-'].sum()
    bucket_sums[bucket_id] = s_b

s_b_df = pd.DataFrame.from_dict(bucket_sums, orient='index', columns=['S_b'])
s_b_df.index.name = 'Bucket'
display_df(s_b_df, "Bucket Sums (S_b)")

# ==============================================================================
# Cell 5: Step 10 - Calculate Cross-Bucket Capital (Medium Scenario)
# ==============================================================================
print("### Step 10: Calculate Cross-Bucket Capital (Medium Scenario) ###\n")

# Sum of squared K_b values
sum_sq_kb = sum(res['K_b']**2 for res in bucket_results.values())

# Vectorized cross-bucket correlation term
s_values = s_b_df['S_b'].values
outer_s = np.outer(s_values, s_values)
psi_s = ~((s_values[:, None] < 0) & (s_values[None, :] < 0))
corr_matrix_s = curvature_gamma_medium * outer_s * psi_s
cross_bucket_corr_term = np.sum(corr_matrix_s) - np.sum(np.diag(corr_matrix_s))


# Final RCCR for medium scenario
rccr_medium = np.sqrt(max(0, sum_sq_kb + cross_bucket_corr_term))

print(f"Sum of Squared K_b's = {sum_sq_kb:,.2f}")
print(f"Cross-Bucket Correlation Term = {cross_bucket_corr_term:,.2f}")
print(f"Final Capital (Medium Scenario) = sqrt({sum_sq_kb:,.0f} + {cross_bucket_corr_term:,.0f}) = {rccr_medium:,.2f}")
print("\n" + "="*80 + "\n")


# ==============================================================================
# Cell 6: Step 11 - Correlation Scenarios & Final Capital
# ==============================================================================
print("### Step 11: Correlation Scenarios & Final Capital ###\n")
print("Recalculating the entire capital charge for Low, Medium, and High correlation scenarios.\n")

def calculate_total_capital_for_scenario(scenario_name):
    """Calculates the total RCCR for a given scenario ('Low', 'Medium', 'High')."""

    # 1. Determine Scenario Correlations
    if scenario_name == 'High':
        rho_scen = curvature_rho_medium * 1.25
        gamma_scen = curvature_gamma_medium * 1.25
    elif scenario_name == 'Low':
        rho_scen = max(2 * curvature_rho_medium - 1, 0.75 * curvature_rho_medium)
        gamma_scen = max(2 * curvature_gamma_medium - 1, 0.75 * curvature_gamma_medium)
    else: # Medium
        rho_scen = curvature_rho_medium
        gamma_scen = curvature_gamma_medium

    # 2. Recalculate Bucket Capitals (K_b)
    scen_bucket_results = {}
    for bucket_id, group in net_positions_df.groupby('Bucket'):
        _, _, k_final, scenario = calculate_bucket_capital_components(group, rho_scen)
        scen_bucket_results[bucket_id] = {'K_b': k_final, 'Selected Scenario': scenario}

    # 3. Recalculate Bucket Sums (S_b)
    scen_bucket_sums = {}
    for bucket_id, result in scen_bucket_results.items():
        bucket_df = net_positions_df[net_positions_df['Bucket'] == bucket_id]
        scen_bucket_sums[bucket_id] = bucket_df['CVR+'].sum() if result['Selected Scenario'] == 'Upward' else bucket_df['CVR-'].sum()

    # 4. Recalculate Final RCCR
    sum_sq_kb_scen = sum(res['K_b']**2 for res in scen_bucket_results.values())

    s_values_scen = np.array(list(scen_bucket_sums.values()))
    outer_s_scen = np.outer(s_values_scen, s_values_scen)
    psi_s_scen = ~((s_values_scen[:, None] < 0) & (s_values_scen[None, :] < 0))
    corr_matrix_s_scen = gamma_scen * outer_s_scen * psi_s_scen
    cross_bucket_corr_term_scen = np.sum(corr_matrix_s_scen) - np.sum(np.diag(corr_matrix_s_scen))

    rccr_final = np.sqrt(max(0, sum_sq_kb_scen + cross_bucket_corr_term_scen))

    return {
        'Intra-Bucket Corr': rho_scen,
        'Cross-Bucket Corr': gamma_scen,
        'Total Capital (RCCR)': rccr_final
    }

# Calculate for all scenarios
scenario_results = {
    'Low': calculate_total_capital_for_scenario('Low'),
    'Medium': calculate_total_capital_for_scenario('Medium'),
    'High': calculate_total_capital_for_scenario('High')
}

scenario_df = pd.DataFrame(scenario_results).T
scenario_df.index.name = "Scenario"
scenario_df['Intra-Bucket Corr'] = scenario_df['Intra-Bucket Corr'].map('{:.4%}'.format)
scenario_df['Cross-Bucket Corr'] = scenario_df['Cross-Bucket Corr'].map('{:.4%}'.format)

display_df(scenario_df, "Scenario Capital Results Summary")

# Determine Final Capital Charge
final_capital_charge = scenario_df['Total Capital (RCCR)'].max()

print("### Final CSR Curvature Capital Requirement ###\n")
print(f"The final capital charge is the maximum of the three scenarios (Article 325h).\n")
print(f"Final Capital = max({scenario_df['Total Capital (RCCR)'].iloc[0]:,.2f}, {scenario_df['Total Capital (RCCR)'].iloc[1]:,.2f}, {scenario_df['Total Capital (RCCR)'].iloc[2]:,.2f})")
print(f"Final Capital Charge = {final_capital_charge:,.2f}")

### Step 3 & 4: Establish Net Positions ###

As per Article 325g(1), CVR values are calculated on a net basis for each risk factor (issuer).

The following DataFrame represents the final net positions for our portfolio.

--- Final Net Positions ---
   Bucket    Issuer  CVR+  CVR-
0       5  Issuer A  -883  1621
1       5  Issuer B -4591  4061
2       5  Issuer C -4358  4421
3       5  Issuer D  -884  1743
4       5  Issuer E  -778  1594
5       6  Issuer F  -384  1091
6       6  Issuer G  -347  1215
7       6  Issuer H  -236   932
8       6  Issuer I -2243  2854


### Steps 5 & 6: Determine Base Curvature Correlations (Medium Scenario) ###

Curvature correlations are the square of the corresponding delta correlations (Article 325ay(5)).

Base Intra-Bucket Delta Correlation (rho_delta): 35.00%
Base Intra-Bucket Curvature Correlation (rho_curvature): 0.1225 or 12.25%

Base Cross-Bucket Delta Correlation (gamma_delta): 25.00%
Base Cross-Bucket Curvature Correlation (gamma_curvature): 0.06